# 03 - Feature Engineering

**Project:** Pump It Up — Data Mining the Water Table
**Author:** Jarret Angbazo

---

### Objective
Transform cleaned data into model-ready feature matrices.
 - Ordinal / binary / one-hot encoding for categoricals
 - Target encoding (LOO-smoothed) for high-cardinality geo columns
 - Interaction features and domain-informed derived features
 - Save final X_train, X_test, y_train for modeling notebooks


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import warnings; warnings.filterwarnings("ignore")

DATA_IN  = Path("../data/processed")
DATA_OUT = Path("../data/processed")


---------------------------------------------------------------------------
1. Load cleaned data
---------------------------------------------------------------------------

In [3]:
train = pd.read_csv(DATA_IN / "train_cleaned.csv")
test  = pd.read_csv(DATA_IN / "test_cleaned.csv")
y     = train["status_group_enc"].astype(int)
y_str = train["status_group"]

print(f"Train: {train.shape}  |  Test: {test.shape}")


Train: (59400, 33)  |  Test: (14850, 31)


---------------------------------------------------------------------------
2. Drop target columns from feature set
---------------------------------------------------------------------------

In [4]:
train = train.drop(columns=["status_group", "status_group_enc"], errors="ignore")


---------------------------------------------------------------------------
3. Drop id (not a feature)
---------------------------------------------------------------------------

In [5]:
test_ids = test["id"].copy()
train = train.drop(columns=["id"], errors="ignore")
test  = test.drop(columns=["id"], errors="ignore")


---------------------------------------------------------------------------
4. Binary / boolean columns
---------------------------------------------------------------------------

In [6]:
bool_map = {"True": 1, "False": 0, "Unknown": -1, "true": 1, "false": 0}
for col in ["public_meeting", "permit"]:
    train[col] = train[col].map(bool_map).fillna(-1).astype(int)
    test[col]  = test[col].map(bool_map).fillna(-1).astype(int)


---------------------------------------------------------------------------
5. Ordinal encoding for low-cardinality nominal columns
   (using integer codes — appropriate for tree-based models)
---------------------------------------------------------------------------

In [7]:
low_card_cats = [
    "basin", "region", "extraction_type", "extraction_type_class",
    "management", "payment", "water_quality", "quantity", "source",
    "source_class", "waterpoint_type", "scheme_management",
    "funder", "installer", "lga", "ward"
]

# Fit on train, apply to both (handle unseen test categories)
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                    encoded_missing_value=-1)
low_card_in_train = [c for c in low_card_cats if c in train.columns]
train[low_card_in_train] = oe.fit_transform(train[low_card_in_train])
test_cols = [c for c in low_card_in_train if c in test.columns]
test[test_cols]  = oe.transform(test[test_cols])

print(f"Ordinal-encoded {len(low_card_in_train)} columns")


Ordinal-encoded 16 columns


---------------------------------------------------------------------------
6. LOO-smoothed target encoding for high-cardinality geo columns
   region_code, district_code are already numeric — keep as-is
   Encode for each class (multi-class → 3 encoding passes)
---------------------------------------------------------------------------

In [8]:
def loo_target_encode(train_col, train_y, test_col, n_min=1, alpha=5):
    """
    Leave-one-out smoothed target encoding for a single column.
    Returns encoded train and test series.
    """
    target = train_y.copy()
    df_enc = pd.DataFrame({"cat": train_col.values, "y": target.values})
    global_mean = df_enc["y"].mean()
    agg = df_enc.groupby("cat")["y"].agg(["sum","count"])

    # LOO train encoding (subtract self from sum/count)
    train_enc = pd.Series(index=df_enc.index, dtype=float)
    for idx in df_enc.index:
        cat = df_enc.loc[idx, "cat"]
        n   = agg.loc[cat, "count"]
        s   = agg.loc[cat, "sum"]
        # LOO
        s_loo = s - df_enc.loc[idx, "y"]
        n_loo = n - 1
        smoothed = (s_loo + alpha * global_mean) / (n_loo + alpha)
        train_enc[idx] = smoothed

    # Test encoding (use all train data)
    test_enc = test_col.map(lambda x: (
        (agg.loc[x, "sum"] + alpha * global_mean) / (agg.loc[x, "count"] + alpha)
        if x in agg.index else global_mean
    ))
    return train_enc, test_enc

# Target encode for functional vs non-functional (binary version of y)
# We'll encode separately for each binary view
geo_high_card = []  # These are already handled via ordinal on lga/ward above

# Keep region_code and district_code as numeric (already integers)
print("Region/district codes kept as numeric.")


Region/district codes kept as numeric.


---------------------------------------------------------------------------
7. Domain interaction features
---------------------------------------------------------------------------
7a. Extraction × payment: certain extraction types with no payment correlate strongly
    with failure (both already ordinally encoded → product is a proxy interaction)

In [9]:
train["extract_x_payment"] = train["extraction_type_class"].astype(float) * train["payment"].astype(float)
test["extract_x_payment"]  = test["extraction_type_class"].astype(float)  * test["payment"].astype(float)

# 7b. Pump age bucket (encoded as ordinal decades)
bins   = [-1, 5, 10, 20, 30, 40, 100]
labels_b = [0, 1, 2, 3, 4, 5]
train["age_bucket"] = pd.cut(train["pump_age"], bins=bins, labels=labels_b).astype(float).fillna(3)
test["age_bucket"]  = pd.cut(test["pump_age"],  bins=bins, labels=labels_b).astype(float).fillna(3)

# 7c. High-flow indicator: amount_tsh > 0 AND quantity == "enough" (already encoded)
# Since quantity is ordinally encoded, we'll use the amount_tsh_missing as proxy
train["has_water_flow"] = (1 - train["amount_tsh_missing"])
test["has_water_flow"]  = (1 - test["amount_tsh_missing"])

# 7d. GPS completeness flag
train["has_gps"] = (train["longitude"] > 0).astype(int)
test["has_gps"]  = (test["longitude"] > 0).astype(int)


---------------------------------------------------------------------------
8. Final feature matrix
---------------------------------------------------------------------------
Ensure all columns are numeric

In [10]:
print("\nColumn dtypes before final check:")
for dtype, cols in train.dtypes.groupby(train.dtypes).groups.items():
    print(f"  {dtype}: {list(cols)[:5]}{'...' if len(cols)>5 else ''}")

# Cast any remaining object cols to float (shouldn't be any)
obj_cols = train.select_dtypes(include="object").columns
if len(obj_cols) > 0:
    print(f"\nWARNING — object cols remaining: {list(obj_cols)}")
    train = train.drop(columns=obj_cols)
    test  = test.drop(columns=[c for c in obj_cols if c in test.columns])

# Align train and test to same columns
shared_cols = [c for c in train.columns if c in test.columns]
X_train = train[shared_cols].astype(float)
X_test  = test[shared_cols].astype(float)

print(f"\nFinal X_train: {X_train.shape}")
print(f"Final X_test : {X_test.shape}")
print(f"Features     : {list(X_train.columns)}")
print(f"Null check   — train: {X_train.isnull().sum().sum()}, test: {X_test.isnull().sum().sum()}")



Column dtypes before final check:
  int64: ['region_code', 'district_code', 'public_meeting', 'permit', 'recorded_year']...
  float64: ['amount_tsh', 'funder', 'gps_height', 'installer', 'longitude']...

Final X_train: (59400, 34)
Final X_test : (14850, 34)
Features     : ['amount_tsh', 'funder', 'gps_height', 'installer', 'longitude', 'latitude', 'basin', 'region', 'region_code', 'district_code', 'lga', 'ward', 'population', 'public_meeting', 'scheme_management', 'permit', 'construction_year', 'extraction_type', 'extraction_type_class', 'management', 'payment', 'water_quality', 'quantity', 'source', 'source_class', 'waterpoint_type', 'recorded_year', 'recorded_month', 'amount_tsh_missing', 'pump_age', 'extract_x_payment', 'age_bucket', 'has_water_flow', 'has_gps']
Null check   — train: 0, test: 0


---------------------------------------------------------------------------
9. Save
---------------------------------------------------------------------------

In [11]:
X_train.to_csv(DATA_OUT / "X_train.csv", index=False)
X_test.to_csv(DATA_OUT  / "X_test.csv",  index=False)
y.to_csv(DATA_OUT / "y_train.csv", index=False, header=True)
y_str.to_csv(DATA_OUT / "y_train_str.csv", index=False, header=True)
test_ids.to_csv(DATA_OUT / "test_ids.csv", index=False, header=True)

print("\n" + "=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"  Original features after cleaning  : 29")
print(f"  Features dropped (target/id)      : 3")
print(f"  New engineered features added     : 5")
print(f"    extract_x_payment (interaction)")
print(f"    age_bucket (ordinal binning)")
print(f"    has_water_flow (flow indicator)")
print(f"    has_gps (location completeness)")
print(f"    amount_tsh_missing (MNAR indicator from cleaning)")
print(f"  Final feature count               : {X_train.shape[1]}")
print("\nSaved: X_train.csv, X_test.csv, y_train.csv, test_ids.csv")



FEATURE ENGINEERING SUMMARY
  Original features after cleaning  : 29
  Features dropped (target/id)      : 3
  New engineered features added     : 5
    extract_x_payment (interaction)
    age_bucket (ordinal binning)
    has_water_flow (flow indicator)
    has_gps (location completeness)
    amount_tsh_missing (MNAR indicator from cleaning)
  Final feature count               : 34

Saved: X_train.csv, X_test.csv, y_train.csv, test_ids.csv
